# BiBo Benchmark Studio

Dual-GPU training dashboard. Configure and launch BiBo vs Qwen3MoE comparison.

**Run all cells below to launch the GUI.**

In [ ]:
!git clone https://github.com/IsNoobgrammer/BiBo.git 2>/dev/null || true
%cd BiBo
!pip install -qU transformers einops wandb bitsandbytes pyyaml liger-kernel datasets
!pip install -e . 2>/dev/null || true

In [ ]:
import os
os.environ["WANDB_SILENT"] = "true"
!wandb login || echo 'Set WANDB_API_KEY env var first'

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import yaml
import threading
import subprocess
import sys
import os
import time
import torch

REPO_ROOT = os.getcwd()
CONFIGS_DIR = os.path.join(REPO_ROOT, "bench", "configs")

# Load all config presets
CONFIGS = {}
for fname in sorted(os.listdir(CONFIGS_DIR)):
    if fname.endswith(".yaml"):
        with open(os.path.join(CONFIGS_DIR, fname)) as f:
            CONFIGS[fname.replace(".yaml", "")] = yaml.safe_load(f)
CONFIG_NAMES = list(CONFIGS.keys())

NUM_GPUS = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)]
GPU_COLORS = ["#1a73e8", "#e8710a"]


# =============================================================
# ANALYTICAL PARAM COUNT (no model creation needed)
# =============================================================
def compute_param_count(cfg):
    """Compute total params analytically from a config dict."""
    m = cfg["model"]
    h = m["hidden_size"]
    i = m["intermediate_size"]
    L = m["num_hidden_layers"]
    nh = m["num_attention_heads"]
    nkv = m["num_key_value_heads"]
    head_dim = h // nh
    vocab = m["vocab_size"]

    embed = vocab * h  # embedding (tied with head)

    # Attention per layer: Q, K, V, O projections + 2 norms
    qkv = h * h + h * (nkv * head_dim) * 2 + h * h
    norms = h * 2  # input_layernorm + post_attention_layernorm
    attn_per_layer = qkv + norms

    # Dense MLP per layer (2 dense layers: first + last)
    dense_per_layer = 3 * i * h  # gate+up+down

    moe_layers = L - 2  # first and last are dense
    mi = m.get("moe_intermediate_size", 768)

    model_type = m.get("type", "bibo")
    is_qwen = model_type in ("qwen3moe", "qwen3_moe")

    if is_qwen:
        num_routed = m.get("num_experts", 8)
        expert_params = 3 * mi * h  # gate+up+down per expert
        routed_total = expert_params * num_routed * moe_layers
        router_total = h * num_routed * moe_layers
    else:
        poly = m.get("polyglu_expert_multiplier", 2)
        pairs = m.get("special_expert_pairs", 1)
        num_polyglu = poly * 3
        expert_params = 3 * mi * h
        routed_total = expert_params * num_polyglu * moe_layers
        # Identity/Zero experts have no params
        num_routed = num_polyglu + pairs * 2
        router_total = h * num_routed * moe_layers

    total = embed + attn_per_layer * L + dense_per_layer * 2 + routed_total + router_total
    return total, {
        "embed": embed,
        "attn": attn_per_layer * L,
        "dense": dense_per_layer * 2,
        "moe_routed": routed_total,
        "moe_router": router_total,
        "moe_layers": moe_layers,
        "num_routed": num_routed,
    }


def format_params(n):
    if n >= 1e6:
        return f"{n/1e6:.2f}M"
    elif n >= 1e3:
        return f"{n/1e3:.1f}K"
    return str(n)


# =============================================================
# GPU PANEL BUILDER
# =============================================================
def build_gpu_panel(gpu_id):
    color = GPU_COLORS[gpu_id % 2]
    gpu_name = GPU_NAMES[gpu_id] if gpu_id < NUM_GPUS else "CPU"

    header = widgets.HTML(
        value=f'<div style="background:{color};color:white;padding:8px 16px;'
              f'border-radius:8px;font-weight:bold;font-size:14px;">
              GPU {gpu_id}: {gpu_name}</div>'
    )

    # Model selector
    model_dd = widgets.Dropdown(
        options=CONFIG_NAMES,
        value=CONFIG_NAMES[gpu_id % len(CONFIG_NAMES)],
        description="Model:",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"),
    )

    # Editable MoE hyperparams
    hidden_sl = widgets.IntSlider(
        value=320, min=128, max=512, step=64,
        description="Hidden:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))
    layers_sl = widgets.IntSlider(
        value=16, min=4, max=32, step=2,
        description="Layers:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))
    heads_sl = widgets.IntSlider(
        value=4, min=1, max=16, step=1,
        description="Heads:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))
    kv_sl = widgets.IntSlider(
        value=2, min=1, max=8, step=1,
        description="KV Heads:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))
    mi_sl = widgets.IntSlider(
        value=768, min=256, max=2048, step=64,
        description="MoE Inter:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))
    topk_sl = widgets.IntSlider(
        value=2, min=1, max=8, step=1,
        description="Top-K:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))

    # BiBo-specific: polyglu multiplier, special pairs
    poly_sl = widgets.IntSlider(
        value=2, min=1, max=6, step=1,
        description="PolyGLU:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))
    special_sl = widgets.IntSlider(
        value=1, min=0, max=4, step=1,
        description="Special:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))

    # Qwen-specific: num_experts
    qwen_experts_sl = widgets.IntSlider(
        value=8, min=2, max=32, step=1,
        description="Experts:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))

    # Param count display
    param_info = widgets.HTML(value="")

    def get_model_cfg():
        """Build a config dict from current widget values."""
        cfg = CONFIGS[model_dd.value]
        m = dict(cfg["model"])  # copy
        m["hidden_size"] = hidden_sl.value
        m["num_hidden_layers"] = layers_sl.value
        m["num_attention_heads"] = heads_sl.value
        m["num_key_value_heads"] = kv_sl.value
        m["moe_intermediate_size"] = mi_sl.value
        m["num_experts_per_tok"] = topk_sl.value

        is_qwen = m.get("type") in ("qwen3moe", "qwen3_moe")
        if is_qwen:
            m["num_experts"] = qwen_experts_sl.value
        else:
            m["polyglu_expert_multiplier"] = poly_sl.value
            m["special_expert_pairs"] = special_sl.value

        # Force head divisibility
        if m["hidden_size"] % m["num_attention_heads"] != 0:
            return None
        if m["num_attention_heads"] % m["num_key_value_heads"] != 0:
            return None

        return {"model": m, "training": cfg.get("training", {}),
                "eval": cfg.get("eval", {}), "logging": cfg.get("logging", {}),
                "hardware": cfg.get("hardware", {})}

    def update_param_info(change=None):
        cfg = get_model_cfg()
        if cfg is None:
            param_info.value = '<div style="background:#3d1f1f;color:#f85149;padding:10px;border-radius:6px;font-family:monospace;font-size:12px;">Heads must divide hidden_size, KV must divide heads</div>'
            return

        m = cfg["model"]
        total, breakdown = compute_param_count(cfg)
        is_qwen = m.get("type") in ("qwen3moe", "qwen3_moe")
        num_routed = breakdown["num_routed"]
        topk = m.get("num_experts_per_tok", 2)
        active_routed = breakdown["moe_routed"] * (topk / num_routed) if num_routed > 0 else 0
        active = breakdown["embed"] + breakdown["attn"] + breakdown["dense"] + active_routed + breakdown["moe_router"]

        if is_qwen:
            expert_desc = f"{num_routed} SwiGLU experts"
        else:
            poly = m.get("polyglu_expert_multiplier", 2)
            pairs = m.get("special_expert_pairs", 1)
            expert_desc = f"{poly}x3 PolyGLU + {pairs}x2 special = {num_routed} routed"

        param_info.value = (
            f'<div style="background:#1e1e1e;color:#d4d4d4;padding:10px;'
            f'border-radius:6px;font-family:monospace;font-size:11px;line-height:1.7;">
            <span style="color:#58a6ff;font-weight:bold;">{m.get("type","?").upper()}</span><br>
            <b>Total:</b> <span style="color:#3fb950;">{format_params(total)}</span> 
            | <b>Active:</b> {format_params(int(active))} 
            | <b>Ratio:</b> {total/max(active,1):.1f}x<br>
            <b>Arch:</b> h={m["hidden_size"]} i={m.get("intermediate_size",1024)} mi={m.get("moe_intermediate_size",768)} 
            L={m["num_hidden_layers"]} H={m["num_attention_heads"]}/{m["num_key_value_heads"]}<br>
            <b>MoE:</b> {expert_desc} | Top-{topk} | {breakdown["moe_layers"]} MoE layers<br>
            <b>SSMax:</b> {m.get("use_ssmax", False)} | 
            <b>Shared:</b> {m.get("use_shared_expert", False)}
            </div>
        )

    # Wire up param count updates
    for w in [model_dd, hidden_sl, layers_sl, heads_sl, kv_sl, mi_sl, topk_sl, poly_sl, special_sl, qwen_experts_sl]:
        w.observe(update_param_info, names="value")

    def on_model_change(change=None):
        cfg = CONFIGS[model_dd.value]
        m = cfg["model"]
        hidden_sl.value = m.get("hidden_size", 320)
        layers_sl.value = m.get("num_hidden_layers", 16)
        heads_sl.value = m.get("num_attention_heads", 4)
        kv_sl.value = m.get("num_key_value_heads", 2)
        mi_sl.value = m.get("moe_intermediate_size", 768)
        topk_sl.value = m.get("num_experts_per_tok", 2)
        poly_sl.value = m.get("polyglu_expert_multiplier", 2)
        special_sl.value = m.get("special_expert_pairs", 1)
        qwen_experts_sl.value = m.get("num_experts", 8)
        wandb_name.value = f"{model_dd.value}_{gpu_id}"
        # Show/hide BiBo vs Qwen fields
        is_qwen = m.get("type") in ("qwen3moe", "qwen3_moe")
        poly_sl.layout.display = "none" if is_qwen else ""
        special_sl.layout.display = "none" if is_qwen else ""
        qwen_experts_sl.layout.display = "" if is_qwen else "none"

    model_dd.observe(on_model_change, names="value")
    on_model_change()
    update_param_info()

    # Training hyperparams
    batch_sl = widgets.IntSlider(value=16, min=2, max=64, step=2,
        description="Batch:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))
    steps_sl = widgets.IntSlider(value=50000, min=1000, max=200000, step=1000,
        description="Steps:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))
    lr_sl = widgets.FloatLogSlider(value=-3.5, base=10, min=-6, max=-2, step=0.1,
        description="LR:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))
    seq_sl = widgets.IntSlider(value=1024, min=256, max=2048, step=256,
        description="Seq Len:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))
    accum_sl = widgets.IntSlider(value=4, min=1, max=16, step=1,
        description="Grad Accum:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))
    eval_sl = widgets.IntSlider(value=500, min=100, max=5000, step=100,
        description="Eval Every:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"))

    # Toggles
    triton_cb = widgets.Checkbox(value=True, description="Triton",
        style={"description_width": "80px"})
    compile_cb = widgets.Checkbox(value=True, description="Compile",
        style={"description_width": "80px"})
    wandb_cb = widgets.Checkbox(value=True, description="WandB",
        style={"description_width": "80px"})
    grad_ckpt_cb = widgets.Checkbox(value=False, description="GradCkpt",
        style={"description_width": "80px"})
    toggles_row = widgets.HBox([triton_cb, compile_cb, wandb_cb, grad_ckpt_cb])

    # WandB name
    wandb_name = widgets.Text(
        value=f"{model_dd.value}_{gpu_id}",
        description="W&B:", style={"description_width": "80px"},
        layout=widgets.Layout(width="95%"),
    )

    # Buttons
    start_btn = widgets.Button(description="▶ Start", button_style="success",
        layout=widgets.Layout(width="48%", height="32px"))
    stop_btn = widgets.Button(description="⏹ Stop", button_style="danger",
        layout=widgets.Layout(width="48%", height="32px"), disabled=True)
    status_html = widgets.HTML(value='<span style="color:#888;">Idle</span>')

    log_output = widgets.Output(
        layout=widgets.Layout(max_height="220px", overflow="auto",
            border="1px solid #333", padding="8px", background="#0d1117"))

    # Build training command
    def build_cmd():
        cfg = get_model_cfg()
        if cfg is None:
            return None

        # Write temp config
        tmp_cfg = os.path.join(REPO_ROOT, f"bench/configs/_tmp_gpu{gpu_id}.yaml")
        with open(tmp_cfg, "w") as f:
            yaml.dump(cfg, f, default_flow_style=False)

        cmd = (
            f"CUDA_VISIBLE_DEVICES={gpu_id} python bench/train.py "
            f"--config {tmp_cfg} "
            f"--batch_size {batch_sl.value} "
            f"--total_steps {steps_sl.value} "
            f"--lr {lr_sl.value:.6f} "
            f"--seq_len {seq_sl.value} "
            f"--grad_accum {accum_sl.value} "
            f"--eval_every {eval_sl.value} "
            f"--wandb_name {wandb_name.value} "
        )
        if not triton_cb.value:
            cmd += " --no_triton"
        if not compile_cb.value:
            cmd += " --no_compile"
        if not wandb_cb.value:
            cmd += " --no_wandb"
        if grad_ckpt_cb.value:
            cmd += " --grad_checkpoint"
        return cmd

    # Process management
    proc_holder = [None]
    thread_holder = [None]

    def start_training(_):
        cmd = build_cmd()
        if cmd is None:
            status_html.value = '<span style="color:#f85149;">Invalid config</span>'
            return
        start_btn.disabled = True
        stop_btn.disabled = False
        status_html.value = '<span style="color:#3fb950;">● Training...</span>'

        def _run():
            with log_output:
                clear_output(wait=True)
                print(f"$ {cmd}")
                print("=" * 50)
                proc = subprocess.Popen(
                    cmd, shell=True, stdout=subprocess.PIPE,
                    stderr=subprocess.STDOUT, text=True,
                    bufsize=1, cwd=REPO_ROOT)
                proc_holder[0] = proc
                for line in iter(proc.stdout.readline, ""):
                    print(line, end="", flush=True)
                proc.wait()
                print(f"\n{'=' * 50}")
                print(f"Exit code: {proc.returncode}")
                proc_holder[0] = None
            start_btn.disabled = False
            stop_btn.disabled = True
            if proc.returncode == 0:
                status_html.value = '<span style="color:#3fb950;">● Done</span>'
            else:
                status_html.value = '<span style="color:#f85149;">● Failed</span>'

        thread_holder[0] = threading.Thread(target=_run, daemon=True)
        thread_holder[0].start()

    def stop_training(_):
        p = proc_holder[0]
        if p:
            p.terminate()
        status_html.value = '<span style="color:#d29922;">● Stopped</span>'
        start_btn.disabled = False
        stop_btn.disabled = True

    start_btn.on_click(start_training)
    stop_btn.on_click(stop_training)

    # Assemble panel
    bibo_fields = widgets.VBox([poly_sl, special_sl])
    qwen_fields = widgets.VBox([qwen_experts_sl])

    panel = widgets.VBox([
        header,
        model_dd,
        param_info,
        widgets.HTML("<hr style='border-color:#333;margin:4px 0;'>"),
        hidden_sl, layers_sl, heads_sl, kv_sl, mi_sl, topk_sl,
        widgets.HBox([bibo_fields, qwen_fields]),
        widgets.HTML("<hr style='border-color:#333;margin:4px 0;'>"),
        batch_sl, steps_sl, lr_sl, seq_sl, accum_sl, eval_sl,
        toggles_row,
        wandb_name,
        widgets.HBox([start_btn, stop_btn]),
        status_html,
        log_output,
    ], layout=widgets.Layout(
        border=f"2px solid {color}", padding="12px",
        border_radius="10px", width="48%"))

    return panel, {
        "start_btn": start_btn, "stop_btn": stop_btn,
        "log_output": log_output, "get_model_cfg": get_model_cfg,
    }


# =============================================================
# BUILD GUI
# =============================================================
title = widgets.HTML(
    value='<div style="text-align:center;padding:16px;">
    <h1 style="margin:0;color:#e6edf3;">🔬 BiBo Benchmark Studio</h1>
    <p style="color:#8b949e;margin:4px 0 0 0;">Dual-GPU training dashboard</p>
    </div>'
)

panel0, ctrls0 = build_gpu_panel(0)
if NUM_GPUS > 1:
    panel1, ctrls1 = build_gpu_panel(1)
else:
    panel1 = widgets.VBox([
        widgets.HTML('<div style="border:2px dashed #333;border-radius:10px;'
            'padding:40px;text-align:center;color:#888;">
            Only 1 GPU detected.<br>Add a second GPU for comparison.</div>')
    ])
    ctrls1 = None

gpu_row = widgets.HBox([panel0, panel1],
    layout=widgets.Layout(justify_content="space-between"))

# Global controls
wandb_project = widgets.Text(value="bibo-bench", description="W&B Project:",
    style={"description_width": "100px"}, layout=widgets.Layout(width="300px"))

launch_btn = widgets.Button(description="🚀 Launch Both", button_style="success",
    layout=widgets.Layout(width="180px", height="36px"))
stop_btn = widgets.Button(description="⏹ Stop Both", button_style="danger",
    layout=widgets.Layout(width="180px", height="36px"))

global_row = widgets.HBox([wandb_project, widgets.Label(""), launch_btn, stop_btn],
    layout=widgets.Layout(justify_content="flex-start", align_items="center", gap="12px"))

def launch_both(_):
    ctrls0["start_btn"].click()
    if ctrls1:
        ctrls1["start_btn"].click()

def stop_both(_):
    ctrls0["stop_btn"].click()
    if ctrls1:
        ctrls1["stop_btn"].click()

launch_btn.on_click(launch_both)
stop_btn.on_click(stop_both)

wandb_link = widgets.HTML(
    value=f'<div style="padding:8px;color:#8b949e;font-size:12px;">'
    f'📊 <a href="https://wandb.ai/ablations-tinycompany-ai/{wandb_project.value}" '
    f'target="_blank" style="color:#58a6ff;">Open W&B Dashboard</a></div>')

def update_wandb_link(change=None):
    proj = wandb_project.value
    wandb_link.value = (
        f'<div style="padding:8px;color:#8b949e;font-size:12px;">'
        f'📊 <a href="https://wandb.ai/ablations-tinycompany-ai/{proj}" '
        f'target="_blank" style="color:#58a6ff;">Open W&B Dashboard</a></div>')
wandb_project.observe(update_wandb_link, names="value")

gui = widgets.VBox([
    title, gpu_row,
    widgets.HTML("<hr style='border-color:#333;margin:8px 0;'>"),
    global_row, wandb_link,
])

display(gui)